# R-tag pipeline — single run walkthrough

This notebook runs the **Python R-tag (RTEG) workflow** for the KB331 sample filter.

**Manual reference:** Jing Yang's SKILL flow in [`../rdsBawTEGAutoFromTemp.il`](../rdsBawTEGAutoFromTemp.il)  
**Documentation:** [`../README.md`](../README.md) · [`../CLAUDE.md`](../CLAUDE.md)

Run all cells top-to-bottom from the `python_code/` directory.

| Step | Section | Module(s) |
|---|---|---|
| 1 | Process inputs | `layermap.py`, `inspect_refs.py` |
| 2 | Selection | `separate.py` |
| 3 | Separation | `prep_resonator_ppd.py` |
| 4 | Setting up | `prep_rteg_frame.py`, `export_gds.py` |
| 5 | Routing | *(planned)* |
| 6 | Verification | *(planned)* |


In [1]:
from __future__ import annotations
import io
import sys
from contextlib import redirect_stdout
from pathlib import Path
import gdstk
import pandas as pd


def resolve_python_code_root() -> Path:
    """Find python_code/ by looking for input_files/ + src/."""
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "input_files").is_dir() and (candidate / "src").is_dir():
            return candidate
    return here


ROOT = resolve_python_code_root()
SRC = ROOT / "src"
INPUT_FILES = ROOT / "input_files"
DRAFT = ROOT / "draft_output"

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

DRAFT.mkdir(parents=True, exist_ok=True)

print(f"Working directory: {ROOT}")
print(f"Input files:       {INPUT_FILES}")
print(f"Draft output:      {DRAFT}")
print(f"Source code:       {SRC}")

Working directory: C:\Users\santosr4\Documents\rTEG Automation\python_code
Input files:       C:\Users\santosr4\Documents\rTEG Automation\python_code\input_files
Draft output:      C:\Users\santosr4\Documents\rTEG Automation\python_code\draft_output
Source code:       C:\Users\santosr4\Documents\rTEG Automation\python_code\src


## Define Inputs Here For Running This Notebook

In [2]:
FILTER = INPUT_FILES / "KB331_N_01.gds" # input filter GDS file
FRAME = INPUT_FILES / "KB331_N_Frame.gds" # die frame
PPD = INPUT_FILES / "GSG_frame.gds" # GSG ppd frame
LAYERMAP = INPUT_FILES / "layermap" # Skyworks layer map

## 1. Process Inputs

In [3]:
# Ensure all referenced input files exist, report and handle missing files

input_files = {
    "Filter layout": FILTER,
    "Frame template": FRAME,
    "Probe device": PPD,
    "Layer table": LAYERMAP,
}

input_roles = {
    "Filter layout": "Clean hierarchical filter GDS — resonators + connect metal",
    "Frame template": None,  # will fill after loading FRAME sizing
    "Probe device": "ppd_1port — GSG pad reference",
    "Layer table": "Skyworks name → GDS (layer, datatype)",
}

file_errors = []

# Check all files, prepare info for table rows
rows = []
frame_size_str = "unknown size"

# Check supporting files first so sizing is available for FRAME
for label, path in input_files.items():
    exists = path.exists()
    if not exists:
        file_errors.append(f"ERROR: {label} file not found: {path}")
    # Calculate size
    size = f"{path.stat().st_size:,} B" if exists and path.is_file() else "—"
    # Frame sizing to be updated after loading below
    rows.append({"file": label, "path": path.name, "exists": exists, "size": size, "role": input_roles[label]})

# Specifically for FRAME, if it is missing, output an error and skip sizing attempts
if FRAME.exists():
    try:
        frame_lib = gdstk.read_gds(FRAME)
        frame_cell = frame_lib.top_level()[0]
        frame_bb = frame_cell.bounding_box()
        if frame_bb is not None:
            (fx0, fy0), (fx1, fy1) = frame_bb
            frame_width = fx1 - fx0
            frame_height = fy1 - fy0
            frame_size_str = f"{frame_width:.1f}×{frame_height:.1f} µm"
    except Exception as e:
        file_errors.append(f"ERROR: Unable to read Frame template or extract bounding box: {e}")

# Update the row for FRAME with computed size string
for row in rows:
    if row["file"] == "Frame template":
        row["role"] = f"{frame_size_str} GSG probe frame (six BAW_MB2 pads)"

# If there were errors, print them, otherwise continue
if file_errors:
    for err in file_errors:
        print(err)
    print("❌ Aborting due to missing required input files ❌")
else:
    display(pd.DataFrame(rows))

,file,path,exists,size,role
0,Filter layout,KB331_N_01.gds,True,"655,360 B",Clean hierarchical filter GDS — resonators + c...
1,Frame template,KB331_N_Frame.gds,True,"34,816 B",460.0×580.0 µm GSG probe frame (six BAW_MB2 pads)
2,Probe device,GSG_frame.gds,True,"10,240 B",ppd_1port — GSG pad reference
3,Layer table,layermap,True,"3,971 B","Skyworks name → GDS (layer, datatype)"


## 2. Selection

2.1 start with layermap definitions

2.2 inspect layer references

2.3 start identifying resonators from filter GDS file

### 2.1 `layermap.py` — layer name lookups

Loads the Skyworks layermap file from `LAYERMAP` (defined above). Maps names like `BAW_MBE` to GDS `(layer, datatype)` pairs.

In [4]:
from src.layermap import load_layermap

layermap = load_layermap(LAYERMAP)
print(f"Loaded {len(layermap)} layers from {LAYERMAP.name}")

for name in ("BAW_MBE", "BAW_MTE", "BAW_MB2", "BAW_EDGE"):
    layer, dt = layermap.pair(name)
    print(f"  {name:12s} -> GDS ({layer}, {dt})")

Loaded 155 layers from layermap
  BAW_MBE      -> GDS (2, 0)
  BAW_MTE      -> GDS (5, 0)
  BAW_MB2      -> GDS (12, 0)
  BAW_EDGE     -> GDS (9, 0)


### 2.2 `inspect_refs.py` — hierarchy sanity check

Lists placed references in the filter GDS: resonators, vias, connect cells. Useful when instance names did not survive export.

In [5]:
from src.inspect_refs import inspect_gds

buf = io.StringIO()
with redirect_stdout(buf):
    inspect_gds(FILTER)

# 
print(buf.getvalue()[:2000])
if len(buf.getvalue()) > 2000:
    print("... (truncated)")


=== KB331_N_01.gds ===

shuntq3_CDNS_781040784740: 80 polys, bbox (-127.4, -49.5)-(128.1, 54.5) (no references)
   labels (9):
     'P1' @ (0.0, -0.0)  layer=(5,0)
     'P2' @ (0.0, -0.0)  layer=(2,0)
     'freq=1478M INFRAver=3.0 ModelID=430 Band=nil multiKt2=None pcType=Q' @ (0.0, 0.0)  layer=(100,0)
     'ORaW=3.4' @ (0.0, 9.5)  layer=(221,0)
     '[@instanceName]' @ (0.0, 36.0)  layer=(221,0)
     'ReW=-99' @ (0.0, 1.5)  layer=(221,0)
     'MRaW=2.2' @ (0.0, -6.5)  layer=(221,0)
     'MTE_CLEN=172.6' @ (0.0, -14.5)  layer=(221,0)
     'MBE_CLEN=377.9' @ (0.0, -22.5)  layer=(221,0)

seriesq3_CDNS_781040784741: 93 polys, bbox (-37.6, -53.2)-(43.3, 53.2) (no references)
   labels (9):
     'P1' @ (0.0, 0.0)  layer=(5,0)
     'P2' @ (0.0, 0.0)  layer=(2,0)
     'freq=1541M INFRAver=3.0 ModelID=430 Band=nil multiKt2=None pcType=Q' @ (0.0, 0.0)  layer=(100,0)
     'ORaW=3.4' @ (0.0, 5.0)  layer=(221,0)
     '[@instanceName]' @ (0.0, 10.0)  layer=(221,0)
     'ReW=1.8' @ (0.0, 0.0)  laye

### 2.3 `separate.py` — resonator and vtb identification

Finds placed resonators (masters: `series`, `shunt`, `rcap`, `mimcap`) and `vtb` vias under the filter parent cell. Returns dataframe-ready rows via `identify(FILTER)`.

In [6]:
from src.separate import identify

identification = identify(FILTER)

parent = identification.parent
filter_lib = identification.library
res_list = identification.resonators
vias = identification.vias

res_df = pd.DataFrame(identification.resonator_rows()) #DF of resonators 
via_df = pd.DataFrame(identification.via_rows()) #DF of VIAs

print(f"Parent cell: {parent}")
print(f"Resonators: {len(res_list)}  |  Vias: {len(vias)}")

res_df

Parent cell: KB331_N_01
Resonators: 8  |  Vias: 4


,index,inst_name,master_name,type,origin_x,origin_y,rotation_deg,split_base
0,0,shuntq3_CDNS_781040784740,shuntq3_CDNS_781040784740,shunt,282.6,183.1,0.0,None
1,1,shuntq3_CDNS_781040784740,shuntq3_CDNS_781040784740,shunt,234.0,98.3,180.0,None
2,2,seriesq3_CDNS_781040784741,seriesq3_CDNS_781040784741,series,95.8,145.1,90.0,None
3,3,seriesq3_CDNS_781040784741,seriesq3_CDNS_781040784741,series,92.0,217.4,270.0,None
4,4,shuntq3_CDNS_781040784742,shuntq3_CDNS_781040784742,shunt,311.9,458.9,90.0,None
5,5,shuntq3_CDNS_781040784745,shuntq3_CDNS_781040784745,shunt,157.8,412.7,0.0,None
6,6,seriesq3_CDNS_781040784747,seriesq3_CDNS_781040784747,series,307.6,317.6,270.0,None
7,7,seriesq3_CDNS_781040784748,seriesq3_CDNS_781040784748,series,187.7,294.1,270.0,None


In [7]:
via_df

,index,master_name,origin_x,origin_y,rotation_deg
0,0,vtb3_CDNS_781040784743,208.6,127.4,270.0
1,1,vtb3_CDNS_781040784746,335.0,158.9,270.0
2,2,vtb3_CDNS_781040784749,122.6,176.9,270.0
3,3,vtb3_CDNS_781040784749,65.2,185.8,90.0


## 3. Separation
For each resonator found place it together with the GSG ppd frame in the center

### `prep_resonator_ppd.py` — PPD + resonator

For each row in `res_df`, combine the resonator with the GSG PPD frame (`GSG_frame.gds`):

1. PPD placed at `(0, 0)`
2. Resonator bbox center aligned to the PPD bbox center
3. If resonator MBE/MTE still overlaps GSG pad metal, nudge the resonator left/right or up/down only until clear

Returns `ppd_assemblies` (in-memory, editable) and `ppd_files_df`. Step 4 (`prep_rteg_frame.py`) places these in the die frame.

In [8]:
from IPython.display import HTML, display
from src.prep_resonator_ppd import (
    assemblies_summary_df,
    prep_resonator_ppd,
    preview_assembly_svg,
)

ppd_assemblies = prep_resonator_ppd(res_df, res_list, PPD)
ppd_files_df = assemblies_summary_df(ppd_assemblies)

print(f"Built {len(ppd_assemblies)} PPD assemblies")
print(f"PPD assembly center: ({ppd_assemblies[0].assembly_center[0]:.1f}, {ppd_assemblies[0].assembly_center[1]:.1f})")
display(ppd_files_df)

Built 8 PPD assemblies
PPD assembly center: (388.2, 245.3)


,index,inst_name,master_name,type,ppd_origin_x,ppd_origin_y,resonator_origin_x,resonator_origin_y,centering_shift_x,centering_shift_y,clearance_shift_x,clearance_shift_y,shift_x,shift_y,assembly_center_x,assembly_center_y
0,0,shuntq3_CDNS_781040784740,shuntq3_CDNS_781040784740,shunt,0.0,0.0,478.6,242.8,105.2,59.7,90.8,0.0,196.0,59.7,388.2,245.3
1,1,shuntq3_CDNS_781040784740,shuntq3_CDNS_781040784740,shunt,0.0,0.0,476.6,242.8,153.8,144.5,88.8,0.0,242.6,144.5,388.2,245.3
2,2,seriesq3_CDNS_781040784741,seriesq3_CDNS_781040784741,series,0.0,0.0,404.4,245.3,289.5,100.2,19.0,0.0,308.6,100.2,388.2,245.3
3,3,seriesq3_CDNS_781040784741,seriesq3_CDNS_781040784741,series,0.0,0.0,402.4,245.3,293.3,27.9,17.0,0.0,310.4,27.9,388.2,245.3
4,4,shuntq3_CDNS_781040784742,shuntq3_CDNS_781040784742,shunt,0.0,0.0,448.7,245.8,89.0,-213.1,47.8,0.0,136.8,-213.1,388.2,245.3
5,5,shuntq3_CDNS_781040784745,shuntq3_CDNS_781040784745,shunt,0.0,0.0,453.7,231.8,230.1,-180.9,65.8,0.0,295.9,-180.9,388.2,245.3
6,6,seriesq3_CDNS_781040784747,seriesq3_CDNS_781040784747,series,0.0,0.0,403.7,245.3,78.8,-72.3,17.3,0.0,96.1,-72.3,388.2,245.3
7,7,seriesq3_CDNS_781040784748,seriesq3_CDNS_781040784748,series,0.0,0.0,429.4,245.6,193.4,-48.5,48.3,0.0,241.7,-48.5,388.2,245.3


simply preview of the resonator + ppd GSG placement within the notebook directly 

In [9]:
# Display/Preview of frame within this notebook

_preview_cols = 4
_preview_items = "\n".join(
    f'<div class="ppd-preview-item">'
    f'<div class="ppd-preview-label">[{asm.index}] {asm.inst_name} ({asm.res_type})</div>'
    f'{preview_assembly_svg(asm)}'
    f"</div>"
    for asm in ppd_assemblies
)
display(
    HTML(
        f"""
<style>
.ppd-preview-grid {{
  display: grid;
  grid-template-columns: repeat({_preview_cols}, minmax(0, 1fr));
  gap: 8px;
  margin-top: 8px;
}}
.ppd-preview-item {{
  border: 1px solid #ccc;
  padding: 4px;
  text-align: center;
  overflow: hidden;
}}
.ppd-preview-label {{
  font-size: 11px;
  margin-bottom: 4px;
}}
.ppd-preview-item svg {{
  width: 100%;
  height: auto;
  display: block;
}}
</style>
<div class="ppd-preview-grid">
{_preview_items}
</div>
"""
    )
)

## 4. Setting Up

place the ppd+resonator combo now into the original die frame.

for the width (x axis) leave 4% margin in both sides

for the height (y axis) leave 7% margin in both sides

### `prep_rteg_frame.py` — die frame placement

For each `ppd_assembly`, place the step-2 `top_cell` inside `FRAME`:

1. Die frame at `(0, 0)`; margins measured from the **inner** MBE die frame (not the outer 460×580 bbox)
2. Assembly left edge at 3.5% inside the inner frame; Y centered with 7% margins
3. MBE filler rectangle on the right: inner-frame center → margined right edge, same height as the placed GSG/resonator assembly

Returns `rteg_assemblies` and `rteg_files_df`. Step 5 (later) adds interconnect routing and DRC.

In [10]:
from src.prep_rteg_frame import (
    assemblies_summary_df,
    prep_rteg_in_frame,
    preview_assembly_svg,
)

rteg_assemblies = prep_rteg_in_frame(ppd_assemblies, FRAME)
rteg_files_df = assemblies_summary_df(rteg_assemblies)

print(f"Built {len(rteg_assemblies)} RTEG frame assemblies")
print(
    f"Content center: ({rteg_assemblies[0].content_center[0]:.1f}, "
    f"{rteg_assemblies[0].content_center[1]:.1f})"
)
display(rteg_files_df)

Built 8 RTEG frame assemblies
Content center: (230.0, 290.0)


,index,inst_name,type,frame_origin_x,frame_origin_y,assembly_origin_x,assembly_origin_y,content_center_x,content_center_y,inner_frame_x0,...,inner_frame_x1,inner_frame_y1,mbe_filler_x0,mbe_filler_y0,mbe_filler_x1,mbe_filler_y1,inner_content_width,mbe_filler_width,x_margin_pct,y_margin_pct
0,0,shuntq3_CDNS_781040784740,shunt,0.0,0.0,-216.2,44.7,230.0,290.0,36.5,...,423.5,543.5,230.0,102.5,408.0,477.5,356.0,178.0,0.04,0.07
1,1,shuntq3_CDNS_781040784740,shunt,0.0,0.0,-216.2,44.7,230.0,290.0,36.5,...,423.5,543.5,230.0,102.5,408.0,477.5,356.0,178.0,0.04,0.07
2,2,seriesq3_CDNS_781040784741,series,0.0,0.0,-216.2,44.7,230.0,290.0,36.5,...,423.5,543.5,230.0,102.5,408.0,477.5,356.0,178.0,0.04,0.07
3,3,seriesq3_CDNS_781040784741,series,0.0,0.0,-216.2,44.7,230.0,290.0,36.5,...,423.5,543.5,230.0,102.5,408.0,477.5,356.0,178.0,0.04,0.07
4,4,shuntq3_CDNS_781040784742,shunt,0.0,0.0,-216.2,44.7,230.0,290.0,36.5,...,423.5,543.5,230.0,102.5,408.0,477.5,356.0,178.0,0.04,0.07
5,5,shuntq3_CDNS_781040784745,shunt,0.0,0.0,-216.2,44.7,230.0,290.0,36.5,...,423.5,543.5,230.0,102.5,408.0,477.5,356.0,178.0,0.04,0.07
6,6,seriesq3_CDNS_781040784747,series,0.0,0.0,-216.2,44.7,230.0,290.0,36.5,...,423.5,543.5,230.0,102.5,408.0,477.5,356.0,178.0,0.04,0.07
7,7,seriesq3_CDNS_781040784748,series,0.0,0.0,-216.2,44.7,230.0,290.0,36.5,...,423.5,543.5,230.0,102.5,408.0,477.5,356.0,178.0,0.04,0.07


In [11]:
from IPython.display import HTML, display

_preview_cols = 4
_preview_items = "\n".join(
    f'<div class="rteg-preview-item">'
    f'<div class="rteg-preview-label">[{asm.index}] {asm.inst_name} ({asm.ppd_assembly.res_type})</div>'
    f'{preview_assembly_svg(asm)}'
    f"</div>"
    for asm in rteg_assemblies
)
display(
    HTML(
        f"""
<style>
.rteg-preview-grid {{
  display: grid;
  grid-template-columns: repeat({_preview_cols}, minmax(0, 1fr));
  gap: 8px;
  margin-top: 8px;
}}
.rteg-preview-item {{
  border: 1px solid #ccc;
  padding: 4px;
  text-align: center;
  overflow: hidden;
}}
.rteg-preview-label {{
  font-size: 11px;
  margin-bottom: 4px;
}}
.rteg-preview-item svg {{
  width: 100%;
  height: auto;
  display: block;
}}
</style>
<div class="rteg-preview-grid">
{_preview_items}
</div>
"""
    )
)

In [12]:
from src.export_gds import export_gds, export_summary_df

rteg_export_df = export_summary_df(
    export_gds(
        rteg_assemblies,
        DRAFT,
        layermap=layermap,
        parent=parent,
        stage="framed",
    )
)

print(f"Exported {len(rteg_export_df)} GDS files to {DRAFT}")
print("Layer names: open each .gds with its matching .lyp in KLayout")
display(rteg_export_df)


Exported 8 GDS files to C:\Users\santosr4\Documents\rTEG Automation\python_code\draft_output
Layer names: open each .gds with its matching .lyp in KLayout


,index,inst_name,cell_name,path,lyp_path,size_bytes
0,0,shuntq3_CDNS_781040784740,rteg_0_shuntq3_CDNS_781040784740,C:\Users\santosr4\Documents\rTEG Automation\py...,C:\Users\santosr4\Documents\rTEG Automation\py...,117336
1,1,shuntq3_CDNS_781040784740,rteg_1_shuntq3_CDNS_781040784740,C:\Users\santosr4\Documents\rTEG Automation\py...,C:\Users\santosr4\Documents\rTEG Automation\py...,117450
2,2,seriesq3_CDNS_781040784741,rteg_2_seriesq3_CDNS_781040784741,C:\Users\santosr4\Documents\rTEG Automation\py...,C:\Users\santosr4\Documents\rTEG Automation\py...,124300
3,3,seriesq3_CDNS_781040784741,rteg_3_seriesq3_CDNS_781040784741,C:\Users\santosr4\Documents\rTEG Automation\py...,C:\Users\santosr4\Documents\rTEG Automation\py...,124300
4,4,shuntq3_CDNS_781040784742,rteg_4_shuntq3_CDNS_781040784742,C:\Users\santosr4\Documents\rTEG Automation\py...,C:\Users\santosr4\Documents\rTEG Automation\py...,135964
5,5,shuntq3_CDNS_781040784745,rteg_5_shuntq3_CDNS_781040784745,C:\Users\santosr4\Documents\rTEG Automation\py...,C:\Users\santosr4\Documents\rTEG Automation\py...,134290
6,6,seriesq3_CDNS_781040784747,rteg_6_seriesq3_CDNS_781040784747,C:\Users\santosr4\Documents\rTEG Automation\py...,C:\Users\santosr4\Documents\rTEG Automation\py...,122492
7,7,seriesq3_CDNS_781040784748,rteg_7_seriesq3_CDNS_781040784748,C:\Users\santosr4\Documents\rTEG Automation\py...,C:\Users\santosr4\Documents\rTEG Automation\py...,120644


========================================================

## 5. Routing - solving interconnect algorithm

now we solve the interconnects layout of each rteg as setup of the rteg frame is all done

## 6. Verification - with clean drc layout

give it a real layout to verify if its accurate   